In [1]:
import subprocess, sys

needs_restart = False
for pkg in [('xgboost', 'xgboost'), ('tensorflow', 'tensorflow')]:
    pkg_name, import_name = pkg
    try:
        __import__(import_name)
        print(f'{import_name} disponible ✅')
    except ImportError:
        print(f'Instalando {pkg_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name, '-q'])
        print(f'{pkg_name} instalado ✅')
        needs_restart = True

if needs_restart:
    print('\n-> Reiniciar kernel: Kernel → Restart Kernel')
    print('-> Luego ejecutar desde Cell 0 hacia abajo.')
else:
    print('Todas las dependencias disponibles ✅ — continuar con Cell 0')


Instalando xgboost...



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3.11 -m pip install --upgrade pip


xgboost instalado ✅


2026-07-28 07:24:55.982959: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-28 07:24:55.983699: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-28 07:24:55.988192: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-28 07:24:55.999786: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785223496.019655     232 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785223496.02

tensorflow disponible ✅

-> Reiniciar kernel: Kernel → Restart Kernel
-> Luego ejecutar desde Cell 0 hacia abajo.


In [1]:
import os, gc, warnings, time, pickle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, confusion_matrix,
    classification_report, cohen_kappa_score
)
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SAVE_DIR     = '/srv/work'
IN_FEAT      = f'{SAVE_DIR}/te_features_v4.parquet'
OUT_DIR      = f'{SAVE_DIR}/models'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
HORIZON      = 2
LABEL_COLS   = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}
ID_COLS      = ['source_agent', 'cloud_provider', 'region', 'window']
TARGET_COL   = 'label_ventana'
SPLIT_COL    = 'split'

UMBRALES = {
    'qwk':         {'floor': 0.65, 'target': 0.72, 'stretch': 0.80},
    'mae_ord':     {'floor': 0.55, 'target': 0.45, 'stretch': 0.35},
    'macro_f1':    {'floor': 0.55, 'target': 0.65, 'stretch': 0.75},
    'bal_acc':     {'floor': 0.60, 'target': 0.70, 'stretch': 0.78},
    'recall_alta': {'floor': 0.55, 'target': 0.75, 'stretch': 0.85},
    'prec_alta':   {'floor': 0.50, 'target': 0.62, 'stretch': 0.75},
}

import sys
if '/srv/python_packages' not in sys.path:
    sys.path.insert(0, '/srv/python_packages')

import mlflow
import mlflow.sklearn
import mlflow.xgboost

MLFLOW_URI        = 'http://localhost:6000'
MLFLOW_EXPERIMENT = 'latam2cloud'

try:
    import urllib.request
    urllib.request.urlopen(f'{MLFLOW_URI}/health', timeout=5)
    print(f'MLflow OK — {MLFLOW_URI}')
except Exception as e:
    print(f'[ADVERTENCIA] MLflow no responde en {MLFLOW_URI}')
    print(f'  Abrir terminal y ejecutar: bash /srv/start_mlflow.sh ')
    print(f'  Error: {e}')

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f'NumPy {np.__version__}  |  Pandas {pd.__version__}  |  XGBoost {xgb.__version__}  |  TensorFlow {tf.__version__}')
print(f'Input: {IN_FEAT}')
print('\nCriterios de éxito (congelados — Benchmarking v2.0, 2026-07-24):')
for m, t in UMBRALES.items():
    print(f'  {m:<14}: Floor={t["floor"]}  Target={t["target"]}  Stretch={t["stretch"]}')


2026-07-28 07:26:15.633127: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-28 07:26:15.634895: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-28 07:26:15.638742: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-28 07:26:15.650436: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785223575.670007     310 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785223575.67

MLflow OK — http://localhost:6000
NumPy 1.26.4  |  Pandas 3.0.3  |  XGBoost 3.2.0  |  TensorFlow 2.19.1
Input: /srv/work/te_features_v4.parquet

Criterios de éxito (congelados — Benchmarking v2.0, 2026-07-24):
  qwk           : Floor=0.65  Target=0.72  Stretch=0.8
  mae_ord       : Floor=0.55  Target=0.45  Stretch=0.35
  macro_f1      : Floor=0.55  Target=0.65  Stretch=0.75
  bal_acc       : Floor=0.6  Target=0.7  Stretch=0.78
  recall_alta   : Floor=0.55  Target=0.75  Stretch=0.85
  prec_alta     : Floor=0.5  Target=0.62  Stretch=0.75


In [2]:
print('Cargando te_features_v4.parquet...')
df = pq.read_table(IN_FEAT).to_pandas()
print(f'Shape: {df.shape}')

df = df.sort_values(ID_COLS)
df['y'] = df.groupby(['source_agent', 'cloud_provider', 'region'])[TARGET_COL].shift(-HORIZON)
df = df.dropna(subset=['y'])
df['y'] = df['y'].astype(int)
print(f'Post target-shift: {df.shape}')

feat_cols_tab = [c for c in df.columns
                 if c not in ID_COLS + [TARGET_COL, SPLIT_COL, 'y']
                 and not c.startswith('n_samp')]
lag_d12_cols     = [c for c in feat_cols_tab
                    if c.startswith('lag48_') or c.startswith('lag336_')]
feat_cols_no_d12 = [c for c in feat_cols_tab if c not in lag_d12_cols]

print(f'\nFeatures tabulares totales : {len(feat_cols_tab)}')
print(f'  Lag D12 (lag48+lag336)   : {len(lag_d12_cols)}')
print(f'  Sin D12 (ablacion)       : {len(feat_cols_no_d12)}')

df_imp = df.copy()
df_imp[feat_cols_tab] = df_imp[feat_cols_tab].fillna(0.0)

std_cols = [c for c in feat_cols_tab if c.startswith('std_')]
bgp_cols = [c for c in feat_cols_tab if c.startswith('bgp_')]
df_xgb   = df.copy()
df_xgb[std_cols] = df_xgb[std_cols].fillna(0.0)
df_xgb[bgp_cols] = df_xgb[bgp_cols].fillna(0.0)

print('\nD17 aplicado:')
print('  df_imp : fillna(0) en TODAS las features  — RF y GRU')
print('  df_xgb : fillna(0) en std_* y bgp_* solamente — XGBoost (lag48/lag336 = NaN nativo)')

train_imp = df_imp[df_imp[SPLIT_COL] == 'train'].reset_index(drop=True)
val_imp   = df_imp[df_imp[SPLIT_COL] == 'val'].reset_index(drop=True)
train_xgb = df_xgb[df_xgb[SPLIT_COL] == 'train'].reset_index(drop=True)
val_xgb   = df_xgb[df_xgb[SPLIT_COL] == 'val'].reset_index(drop=True)

X_train_imp = train_imp[feat_cols_tab].values.astype('float32')
y_train     = train_imp['y'].values.astype(int)
X_val_imp   = val_imp[feat_cols_tab].values.astype('float32')
y_val       = val_imp['y'].values.astype(int)

X_train_xgb = train_xgb[feat_cols_tab].values.astype('float32')
X_val_xgb   = val_xgb[feat_cols_tab].values.astype('float32')

classes      = np.array([0, 1, 2, 3])
class_counts = np.bincount(y_train)
n_total      = len(y_train)
sqrt_w_arr   = np.sqrt(n_total / (4 * class_counts)).astype('float32')
sqrt_w_dict  = dict(zip(classes, sqrt_w_arr))
sample_weights    = np.array([sqrt_w_dict[y] for y in y_train], dtype='float32')
class_weights_rf  = dict(zip(classes,
    compute_class_weight('balanced', classes=classes, y=y_train)))

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_val_sc   = scaler.transform(X_val_imp)
with open(f'{OUT_DIR}/scaler_v4.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f'Scaler v4 guardado: {OUT_DIR}/scaler_v4.pkl')

print(f'\nX_train_imp: {X_train_imp.shape},  X_val_imp: {X_val_imp.shape}')
print(f'X_train_xgb: {X_train_xgb.shape},  X_val_xgb: {X_val_xgb.shape}')
print('\nDistribucion y_train:')
for k in classes:
    n = (y_train == k).sum()
    print(f'  {LABEL_COLS[k]:8s} ({k}): {n:>8,}  ({100*n/n_total:.1f}%)')
print('\nPesos sqrt (clase imbalance):')
for cls in classes:
    print(f'  {LABEL_COLS[cls]:6s}: w_sqrt={sqrt_w_arr[cls]:.2f}  w_rf={class_weights_rf[cls]:.2f}')


Cargando te_features_v4.parquet...
Shape: (3763917, 122)
Post target-shift: (3761847, 123)

Features tabulares totales : 113
  Lag D12 (lag48+lag336)   : 42
  Sin D12 (ablacion)       : 71

D17 aplicado:
  df_imp : fillna(0) en TODAS las features  — RF y GRU
  df_xgb : fillna(0) en std_* y bgp_* solamente — XGBoost (lag48/lag336 = NaN nativo)
Scaler v4 guardado: /srv/work/models/scaler_v4.pkl

X_train_imp: (2125778, 113),  X_val_imp: (644690, 113)
X_train_xgb: (2125778, 113),  X_val_xgb: (644690, 113)

Distribucion y_train:
  Normal   (0): 1,899,347  (89.3%)
  Baja     (1):  153,319  (7.2%)
  Media    (2):   36,421  (1.7%)
  Alta     (3):   36,691  (1.7%)

Pesos sqrt (clase imbalance):
  Normal: w_sqrt=0.53  w_rf=0.28
  Baja  : w_sqrt=1.86  w_rf=3.47
  Media : w_sqrt=3.82  w_rf=14.59
  Alta  : w_sqrt=3.81  w_rf=14.48


In [3]:

def nivel_umbral(valor, metrica, menor_mejor=False):
    t = UMBRALES.get(metrica, {})
    if not t:
        return ''
    if menor_mejor:
        if valor <= t['stretch']: return 'Stretch'
        if valor <= t['target']:  return 'Target'
        if valor <= t['floor']:   return 'Floor'
        return 'Bajo Floor'
    else:
        if valor >= t['stretch']: return 'Stretch'
        if valor >= t['target']:  return 'Target'
        if valor >= t['floor']:   return 'Floor'
        return 'Bajo Floor'

def eval_model(y_true, y_pred, nombre):
    macro_f1    = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    bal_acc     = balanced_accuracy_score(y_true, y_pred)
    qwk         = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    mae_ord     = np.mean(np.abs(np.array(y_true) - np.array(y_pred)))
    cm          = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])

    tp_alta     = cm[3, 3]
    fn_alta     = cm[3, :3].sum()
    fp_alta     = cm[:3, 3].sum()
    recall_alta = tp_alta / (tp_alta + fn_alta) if (tp_alta + fn_alta) > 0 else 0.0
    prec_alta   = tp_alta / (tp_alta + fp_alta) if (tp_alta + fp_alta) > 0 else 0.0

    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  {nombre}')
    print(sep)
    print(f'  Macro-F1         : {macro_f1:.4f}  [{nivel_umbral(macro_f1, "macro_f1")}]')
    print(f'  Weighted-F1      : {weighted_f1:.4f}')
    print(f'  Balanced Accuracy: {bal_acc:.4f}  [{nivel_umbral(bal_acc, "bal_acc")}]')
    print(f'  QWK (ordinal)    : {qwk:.4f}  [{nivel_umbral(qwk, "qwk")}]')
    print(f'  MAE ordinal      : {mae_ord:.4f}  [{nivel_umbral(mae_ord, "mae_ord", menor_mejor=True)}]')
    print(f'  Recall Alta      : {recall_alta:.4f}  [{nivel_umbral(recall_alta, "recall_alta")}]')
    print(f'  Precision Alta   : {prec_alta:.4f}  [{nivel_umbral(prec_alta, "prec_alta")}]')
    candidato = (qwk >= UMBRALES['qwk']['floor'] and recall_alta >= UMBRALES['recall_alta']['floor'])
    print(f'  -> Candidato (QWK>={UMBRALES["qwk"]["floor"]} AND '
          f'RecAlta>={UMBRALES["recall_alta"]["floor"]}): {"SI" if candidato else "NO"}')
    print('\n  Matriz de confusion (filas=real, cols=pred):')
    cm_df = pd.DataFrame(cm,
                         index=['  Normal','  Baja','  Media','  Alta'],
                         columns=['Normal','Baja','Media','Alta'])
    print(cm_df.to_string())
    print('\n  Reporte por clase:')
    print(classification_report(y_true, y_pred,
          target_names=['Normal','Baja','Media','Alta'], zero_division=0))
    return {
        'modelo': nombre, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
        'bal_acc': bal_acc, 'qwk': qwk, 'mae_ord': mae_ord,
        'recall_alta': recall_alta, 'prec_alta': prec_alta,
        'candidato_crit': candidato,
    }

resultados_v3_ref = {
    'baseline_v3': {'macro_f1': 0.6255, 'bal_acc': 0.6254, 'qwk': 0.6149, 'mae_ord': 0.1655, 'recall_alta': 0.5598},
    'rf_v3':       {'macro_f1': 0.6436, 'bal_acc': 0.6533, 'qwk': 0.6324, 'mae_ord': 0.1620, 'recall_alta': 0.6361},
    'xgb_v3':      {'macro_f1': 0.6318, 'bal_acc': 0.6968, 'qwk': 0.6012, 'mae_ord': 0.1956, 'recall_alta': 0.6990},
    'lstm_v3':     {'macro_f1': 0.5984, 'bal_acc': 0.6371, 'qwk': 0.5386, 'mae_ord': 0.2089, 'recall_alta': 0.5741},
}
resultados_v4 = {}
print('Funciones de evaluacion v4 listas.')
print('Resultados v3 cargados como referencia historica.')


Funciones de evaluacion v4 listas.
Resultados v3 cargados como referencia historica.


In [4]:
y_pred_base = val_imp[TARGET_COL].values.astype(int)
resultados_v4['baseline_v4'] = eval_model(y_val, y_pred_base,
                                           'Baseline Estatico v4 — Validacion')

with mlflow.start_run(run_name='baseline_estatico_v4'):
    mlflow.log_param('model_type',      'static_threshold')
    mlflow.log_param('dataset_version', 'v4')
    r = resultados_v4['baseline_v4']
    mlflow.log_metric('val_macro_f1',    r['macro_f1'])
    mlflow.log_metric('val_bal_acc',     r['bal_acc'])
    mlflow.log_metric('val_qwk',         r['qwk'])
    mlflow.log_metric('val_mae_ordinal', r['mae_ord'])
    mlflow.log_metric('val_recall_alta', r['recall_alta'])
    mlflow.log_metric('val_prec_alta',   r['prec_alta'])
    mlflow.log_metric('val_candidato',   int(r['candidato_crit']))
print('Baseline v4 registrado en MLflow.')



  Baseline Estatico v4 — Validacion
  Macro-F1         : 0.6255  [Floor]
  Weighted-F1      : 0.8910
  Balanced Accuracy: 0.6254  [Floor]
  QWK (ordinal)    : 0.6149  [Bajo Floor]
  MAE ordinal      : 0.1655  [Stretch]
  Recall Alta      : 0.5598  [Floor]
  Precision Alta   : 0.5600  [Floor]
  -> Candidato (QWK>=0.65 AND RecAlta>=0.55): NO

  Matriz de confusion (filas=real, cols=pred):
        Normal   Baja  Media   Alta
Normal  521114  18853   5198   6026
Baja     18910  36552   2149   1034
Media     5282   2205   5509   1773
Alta      5946    978   1917  11244

  Reporte por clase:
              precision    recall  f1-score   support

      Normal       0.95      0.95      0.95    551191
        Baja       0.62      0.62      0.62     58645
       Media       0.37      0.37      0.37     14769
        Alta       0.56      0.56      0.56     20085

    accuracy                           0.89    644690
   macro avg       0.63      0.63      0.63    644690
weighted avg       0.89    

In [5]:
print(f'Entrenando RF v4 FULL ({len(feat_cols_tab)} features)...')
print(f'  n_estimators=100  |  class_weight=balanced  |  max_depth=30  |  n_jobs=4')

t0 = time.time()
rf_v4 = RandomForestClassifier(
    n_estimators     = 100,
    max_depth        = 30,
    min_samples_leaf = 5,
    class_weight     = 'balanced',
    n_jobs           = 4,
    random_state     = RANDOM_STATE,
    verbose          = 1,
)
rf_v4.fit(X_train_imp, y_train)
elapsed = time.time() - t0
print(f'\nEntrenamiento: {elapsed/60:.1f} min')

y_pred_rf_v4 = rf_v4.predict(X_val_imp)
resultados_v4['rf_v4_full'] = eval_model(y_val, y_pred_rf_v4, 'RF v4 Full — Validacion')

importances_rf_v4 = pd.Series(rf_v4.feature_importances_, index=feat_cols_tab)
print('\nTop 20 features RF v4 (MDI):')
for feat, imp in importances_rf_v4.sort_values(ascending=False).head(20).items():
    print(f'  {feat:<42s} {imp:.4f}')

imp_lag48_max  = importances_rf_v4[[c for c in lag_d12_cols if c.startswith('lag48_')]].max()
imp_lag336_max = importances_rf_v4[[c for c in lag_d12_cols if c.startswith('lag336_')]].max()
thr_ind = 1.0 / len(feat_cols_tab)
print(f'\nD12-REV preview — MDI max  (umbral individual={thr_ind:.4f}):')
print(f'  lag48_*  max MDI: {imp_lag48_max:.4f}  -> {"H1 (aporta)" if imp_lag48_max >= thr_ind else "H0 (no aporta)"}')
print(f'  lag336_* max MDI: {imp_lag336_max:.4f}  -> {"H1 (aporta)" if imp_lag336_max >= thr_ind else "H0 (no aporta)"}')

with open(f'{OUT_DIR}/rf_v4_full_model.pkl', 'wb') as f:
    pickle.dump(rf_v4, f)
print(f'\nModelo guardado: {OUT_DIR}/rf_v4_full_model.pkl')

print('\n--- Delta RF v4 full vs RF v3 ---')
for m in ['macro_f1', 'bal_acc', 'qwk', 'mae_ord', 'recall_alta']:
    v3 = resultados_v3_ref['rf_v3'].get(m, 0)
    v4 = resultados_v4['rf_v4_full'].get(m, 0)
    delta = v4 - v3
    print(f'  {m:<14}: v3={v3:.4f}  v4={v4:.4f}  ({delta:+.4f})')

with mlflow.start_run(run_name='rf_v4_full'):
    mlflow.log_param('model_type',        'random_forest')
    mlflow.log_param('n_estimators',      100)
    mlflow.log_param('max_depth',         30)
    mlflow.log_param('min_samples_leaf',  5)
    mlflow.log_param('class_weight',      'balanced')
    mlflow.log_param('dataset_version',   'v4')
    mlflow.log_param('n_features',        len(feat_cols_tab))
    mlflow.log_param('d17_fillna',        'all_features_0')
    mlflow.log_param('d20_run',           'full')
    r = resultados_v4['rf_v4_full']
    mlflow.log_metric('val_macro_f1',    r['macro_f1'])
    mlflow.log_metric('val_bal_acc',     r['bal_acc'])
    mlflow.log_metric('val_qwk',         r['qwk'])
    mlflow.log_metric('val_mae_ordinal', r['mae_ord'])
    mlflow.log_metric('val_recall_alta', r['recall_alta'])
    mlflow.log_metric('val_prec_alta',   r['prec_alta'])
    mlflow.log_metric('val_candidato',   int(r['candidato_crit']))
    mlflow.log_param('model_local_path', f'{OUT_DIR}/rf_v4_full_model.pkl')
print('RF v4 full registrado en MLflow.')


Entrenando RF v4 FULL (113 features)...
  n_estimators=100  |  class_weight=balanced  |  max_depth=30  |  n_jobs=4


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  5.7min
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed: 13.1min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.



Entrenamiento: 13.1 min


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.2s finished



  RF v4 Full — Validacion
  Macro-F1         : 0.6388  [Floor]
  Weighted-F1      : 0.8943
  Balanced Accuracy: 0.6471  [Floor]
  QWK (ordinal)    : 0.6257  [Bajo Floor]
  MAE ordinal      : 0.1647  [Stretch]
  Recall Alta      : 0.6466  [Floor]
  Precision Alta   : 0.5464  [Floor]
  -> Candidato (QWK>=0.65 AND RecAlta>=0.55): NO

  Matriz de confusion (filas=real, cols=pred):
        Normal   Baja  Media   Alta
Normal  521230  16900   5587   7474
Baja     18812  36777   1673   1383
Media     5191   2202   5452   1924
Alta      5120    564   1414  12987

  Reporte por clase:
              precision    recall  f1-score   support

      Normal       0.95      0.95      0.95    551191
        Baja       0.65      0.63      0.64     58645
       Media       0.39      0.37      0.38     14769
        Alta       0.55      0.65      0.59     20085

    accuracy                           0.89    644690
   macro avg       0.63      0.65      0.64    644690
weighted avg       0.89      0.89    

In [6]:
del rf_v4
del X_train_sc, X_val_sc
gc.collect()
print('Memoria liberada: rf_v4, X_train_sc, X_val_sc')

print(f'Entrenando RF v4 SIN D12 ({len(feat_cols_no_d12)} features)...')
print(f'  Excluidos: {len(lag_d12_cols)} features lag48_*/lag336_*')

X_tr_nd = train_imp[feat_cols_no_d12].values.astype('float32')
X_vl_nd = val_imp[feat_cols_no_d12].values.astype('float32')

t0 = time.time()
rf_v4_nd = RandomForestClassifier(
    n_estimators     = 100,
    max_depth        = 30,
    min_samples_leaf = 5,
    class_weight     = 'balanced',
    n_jobs           = 4,
    random_state     = RANDOM_STATE,
    verbose          = 1,
)
rf_v4_nd.fit(X_tr_nd, y_train)
elapsed = time.time() - t0
print(f'\nEntrenamiento: {elapsed/60:.1f} min')

y_pred_rf_nd = rf_v4_nd.predict(X_vl_nd)
resultados_v4['rf_v4_no_d12'] = eval_model(y_val, y_pred_rf_nd,
                                             'RF v4 Sin-D12 — Validacion (D12-REV)')

print('\n--- Efecto D12: full vs sin-D12 ---')
for m in ['macro_f1', 'bal_acc', 'qwk', 'mae_ord', 'recall_alta']:
    full  = resultados_v4['rf_v4_full'].get(m, 0)
    nod12 = resultados_v4['rf_v4_no_d12'].get(m, 0)
    delta = full - nod12
    print(f'  {m:<14}: sin-D12={nod12:.4f}  full={full:.4f}  (delta D12={delta:+.4f})')

with open(f'{OUT_DIR}/rf_v4_nod12_model.pkl', 'wb') as f:
    pickle.dump(rf_v4_nd, f)
del rf_v4_nd
gc.collect()
print(f'\nModelo guardado y liberado de memoria.')

with mlflow.start_run(run_name='rf_v4_no_d12'):
    mlflow.log_param('model_type',        'random_forest')
    mlflow.log_param('n_estimators',      100)
    mlflow.log_param('max_depth',         30)
    mlflow.log_param('min_samples_leaf',  5)
    mlflow.log_param('class_weight',      'balanced')
    mlflow.log_param('dataset_version',   'v4')
    mlflow.log_param('n_features',        len(feat_cols_no_d12))
    mlflow.log_param('d17_fillna',        'all_features_0')
    mlflow.log_param('d20_run',           'no_d12')
    r = resultados_v4['rf_v4_no_d12']
    mlflow.log_metric('val_macro_f1',    r['macro_f1'])
    mlflow.log_metric('val_bal_acc',     r['bal_acc'])
    mlflow.log_metric('val_qwk',         r['qwk'])
    mlflow.log_metric('val_mae_ordinal', r['mae_ord'])
    mlflow.log_metric('val_recall_alta', r['recall_alta'])
    mlflow.log_metric('val_prec_alta',   r['prec_alta'])
    mlflow.log_metric('val_candidato',   int(r['candidato_crit']))
    mlflow.log_param('model_local_path', f'{OUT_DIR}/rf_v4_nod12_model.pkl')
print('RF v4 sin-D12 registrado en MLflow.')


Memoria liberada: rf_v4, X_train_sc, X_val_sc
Entrenando RF v4 SIN D12 (71 features)...
  Excluidos: 42 features lag48_*/lag336_*


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  4.4min
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed: 10.5min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.



Entrenamiento: 10.6 min


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    4.7s finished



  RF v4 Sin-D12 — Validacion (D12-REV)
  Macro-F1         : 0.6436  [Floor]
  Weighted-F1      : 0.8950
  Balanced Accuracy: 0.6589  [Floor]
  QWK (ordinal)    : 0.6296  [Bajo Floor]
  MAE ordinal      : 0.1653  [Stretch]
  Recall Alta      : 0.6516  [Floor]
  Precision Alta   : 0.5393  [Floor]
  -> Candidato (QWK>=0.65 AND RecAlta>=0.55): NO

  Matriz de confusion (filas=real, cols=pred):
        Normal   Baja  Media   Alta
Normal  519738  17642   6011   7800
Baja     17918  37445   1849   1433
Media     4792   2083   5947   1947
Alta      4858    557   1582  13088

  Reporte por clase:
              precision    recall  f1-score   support

      Normal       0.95      0.94      0.95    551191
        Baja       0.65      0.64      0.64     58645
       Media       0.39      0.40      0.39     14769
        Alta       0.54      0.65      0.59     20085

    accuracy                           0.89    644690
   macro avg       0.63      0.66      0.64    644690
weighted avg       0.90 

In [7]:
print('Entrenando XGBoost v4...')
print(f'  X_train: {X_train_xgb.shape}  |  n_estimators=500  |  lr=0.05  |  colsample=0.6')

t0 = time.time()
xgb_v4 = xgb.XGBClassifier(
    n_estimators          = 500,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.6,
    objective             = 'multi:softmax',
    num_class             = 4,
    eval_metric           = 'mlogloss',
    early_stopping_rounds = 50,
    n_jobs                = 4,
    random_state          = RANDOM_STATE,
    verbosity             = 1,
    tree_method           = 'hist',
)
xgb_v4.fit(
    X_train_xgb, y_train,
    sample_weight = sample_weights,
    eval_set      = [(X_val_xgb, y_val)],
    verbose       = 50,
)
elapsed = time.time() - t0
print(f'\nTiempo: {elapsed/60:.1f} min  |  best_iteration: {xgb_v4.best_iteration}')

xgb_v4.get_booster().feature_names = feat_cols_tab
y_pred_xgb_v4 = xgb_v4.predict(X_val_xgb)
resultados_v4['xgb_v4'] = eval_model(y_val, y_pred_xgb_v4, 'XGBoost v4 — Validacion')

imp_xgb_v4 = (pd.Series(xgb_v4.get_booster().get_score(importance_type='gain'))
               .sort_values(ascending=False))
print('\nTop 20 features XGBoost v4 (gain):')
print(imp_xgb_v4.head(20).to_string())

xgb_v4.save_model(f'{OUT_DIR}/xgb_v4_model.json')
print(f'\nModelo guardado: {OUT_DIR}/xgb_v4_model.json')

print('\n--- Delta XGB v4 vs v3 ---')
for m in ['macro_f1', 'bal_acc', 'qwk', 'mae_ord', 'recall_alta']:
    v3 = resultados_v3_ref['xgb_v3'].get(m, 0)
    v4 = resultados_v4['xgb_v4'].get(m, 0)
    delta = v4 - v3
    print(f'  {m:<14}: v3={v3:.4f}  v4={v4:.4f}  ({delta:+.4f})')

with mlflow.start_run(run_name='xgb_v4'):
    mlflow.log_param('model_type',           'xgboost')
    mlflow.log_param('n_estimators_best',    xgb_v4.best_iteration + 1)
    mlflow.log_param('max_depth',            6)
    mlflow.log_param('learning_rate',        0.05)
    mlflow.log_param('colsample_bytree',     0.6)
    mlflow.log_param('sample_weight',        'sqrt')
    mlflow.log_param('dataset_version',      'v4')
    mlflow.log_param('n_features',           len(feat_cols_tab))
    mlflow.log_param('d17_nan_lag_d12',      'native_xgb')
    mlflow.log_param('d19_colsample',        0.6)
    r = resultados_v4['xgb_v4']
    mlflow.log_metric('val_macro_f1',    r['macro_f1'])
    mlflow.log_metric('val_bal_acc',     r['bal_acc'])
    mlflow.log_metric('val_qwk',         r['qwk'])
    mlflow.log_metric('val_mae_ordinal', r['mae_ord'])
    mlflow.log_metric('val_recall_alta', r['recall_alta'])
    mlflow.log_metric('val_prec_alta',   r['prec_alta'])
    mlflow.log_metric('val_candidato',   int(r['candidato_crit']))
    mlflow.log_param('model_local_path', f'{OUT_DIR}/xgb_v4_model.json')
print('XGBoost v4 registrado en MLflow.')


Entrenando XGBoost v4...
  X_train: (2125778, 113)  |  n_estimators=500  |  lr=0.05  |  colsample=0.6
[0]	validation_0-mlogloss:0.65131
[50]	validation_0-mlogloss:0.37807
[100]	validation_0-mlogloss:0.35651
[150]	validation_0-mlogloss:0.35157
[200]	validation_0-mlogloss:0.34850
[250]	validation_0-mlogloss:0.34589
[300]	validation_0-mlogloss:0.34370
[350]	validation_0-mlogloss:0.34160
[400]	validation_0-mlogloss:0.34018
[450]	validation_0-mlogloss:0.33890
[499]	validation_0-mlogloss:0.33767

Tiempo: 9.2 min  |  best_iteration: 499

  XGBoost v4 — Validacion
  Macro-F1         : 0.6388  [Floor]
  Weighted-F1      : 0.8879
  Balanced Accuracy: 0.6983  [Floor]
  QWK (ordinal)    : 0.6141  [Bajo Floor]
  MAE ordinal      : 0.1877  [Stretch]
  Recall Alta      : 0.6934  [Floor]
  Precision Alta   : 0.4871  [Bajo Floor]
  -> Candidato (QWK>=0.65 AND RecAlta>=0.55): NO

  Matriz de confusion (filas=real, cols=pred):
        Normal   Baja  Media   Alta
Normal  506796  24037   9214  11144
Baja  

In [4]:

for _var in ['df', 'df_imp', 'df_xgb',
             'X_train_imp', 'X_val_imp', 'X_train_xgb', 'X_val_xgb',
             'X_train_sc', 'X_val_sc',
             'train_xgb', 'val_xgb',
             'xgb_v4', 'sample_weights']:
    if _var in globals():
        del globals()[_var]
gc.collect()
print('Memoria liberada para GRU.')

SEQ_LEN       = 4
FEAT_GRU      = feat_cols_tab
route_keys    = ['source_agent', 'cloud_provider', 'region']
MAX_SEQ_TRAIN = 400_000
MAX_SEQ_VAL   = 200_000

def build_sequences_gru(df_split, seq_len, feat_cols, scaler_, max_samples=None):
    groups = [(key, grp.sort_values('window'))
              for key, grp in df_split.groupby(route_keys, sort=False)
              if len(grp) >= seq_len]
    total = sum(len(grp) - seq_len for _, grp in groups)
    n_out = min(total, max_samples) if max_samples else total
    X_out = np.empty((n_out, seq_len, len(feat_cols)), dtype='float32')
    y_out = np.empty(n_out, dtype='int32')
    idx = 0
    for _, grp in groups:
        vals  = scaler_.transform(grp[feat_cols].values.astype('float32'))
        y_arr = grp['y'].values.astype('int32')
        n     = len(grp)
        for i in range(n - seq_len):
            X_out[idx] = vals[i:i + seq_len]
            y_out[idx] = y_arr[i + seq_len]
            idx += 1
            if idx >= n_out:
                return X_out, y_out
    return X_out[:idx], y_out[:idx]

print(f'Construyendo secuencias train (max={MAX_SEQ_TRAIN:,})...')
X_tr_seq, y_tr_seq = build_sequences_gru(train_imp, SEQ_LEN, FEAT_GRU, scaler,
                                          max_samples=MAX_SEQ_TRAIN)
print(f'  X_train_seq: {X_tr_seq.shape}')

print(f'Construyendo secuencias val (max={MAX_SEQ_VAL:,})...')
X_vl_seq, y_vl_seq = build_sequences_gru(val_imp, SEQ_LEN, FEAT_GRU, scaler,
                                          max_samples=MAX_SEQ_VAL)
print(f'  X_val_seq:   {X_vl_seq.shape}')
gc.collect()


class QWKLoss(keras.losses.Loss):
    """QWK Loss — de la Torre et al. (2018). Minimizar = maximizar QWK."""
    def __init__(self, n_classes=4, alpha=None, **kwargs):
        super().__init__(**kwargs)
        w = np.zeros((n_classes, n_classes), dtype=np.float32)
        for i in range(n_classes):
            for j in range(n_classes):
                w[i, j] = (i - j) ** 2 / (n_classes - 1) ** 2
        self.W     = tf.constant(w, dtype=tf.float32)
        self.n     = n_classes
        self.alpha = tf.constant(alpha, dtype=tf.float32) if alpha is not None else None

    def call(self, y_true, y_pred):
        y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=self.n)
        if self.alpha is not None:
            w_s      = tf.reduce_sum(y_true_oh * self.alpha, axis=1, keepdims=True)
            y_true_w = y_true_oh * w_s
        else:
            y_true_w = y_true_oh
        B   = tf.cast(tf.shape(y_true)[0], tf.float32)
        O   = tf.matmul(tf.transpose(y_true_w), y_pred) / B
        h_t = tf.reduce_sum(y_true_w, axis=0, keepdims=True) / B
        h_p = tf.reduce_mean(y_pred,  axis=0, keepdims=True)
        E   = tf.matmul(tf.transpose(h_t), h_p)
        return tf.reduce_sum(self.W * O) / (tf.reduce_sum(self.W * E) + 1e-9)


def build_gru_model(seq_len, n_features, n_classes=4):
    inp = keras.Input(shape=(seq_len, n_features))
    x   = layers.GRU(64, return_sequences=True,  dropout=0.2)(inp)
    x   = layers.GRU(32, return_sequences=False, dropout=0.2)(x)
    x   = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    return keras.Model(inp, out)


alpha_gru = sqrt_w_arr / sqrt_w_arr.sum()
tf.random.set_seed(RANDOM_STATE)
gru_v4 = build_gru_model(SEQ_LEN, len(FEAT_GRU))
gru_v4.summary()

gru_v4.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-3),
    loss      = QWKLoss(n_classes=4, alpha=alpha_gru),
    metrics   = ['accuracy'],
)

callbacks_v4 = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-5, verbose=1),
    keras.callbacks.ModelCheckpoint(
        filepath=f'{OUT_DIR}/gru_v4_best.keras',
        monitor='val_loss', save_best_only=True, verbose=0),
]

t0 = time.time()
history_gru = gru_v4.fit(
    X_tr_seq, y_tr_seq,
    validation_data = (X_vl_seq, y_vl_seq),
    epochs     = 50,
    batch_size = 512,
    callbacks  = callbacks_v4,
    verbose    = 1,
)
elapsed = time.time() - t0
print(f'\nTiempo: {elapsed/60:.1f} min  |  Epochs: {len(history_gru.history["loss"])}')

y_prob_gru = gru_v4.predict(X_vl_seq, batch_size=2048, verbose=0)
y_pred_gru = np.argmax(y_prob_gru, axis=1)
resultados_v4['gru_v4_qwk'] = eval_model(y_vl_seq, y_pred_gru, 'GRU v4 QWK Loss — Validacion')

gru_v4.save(f'{OUT_DIR}/gru_v4_model.keras')
print(f'Modelo guardado: {OUT_DIR}/gru_v4_model.keras')

print('\n--- Delta GRU v4 vs LSTM v3 ---')
for m in ['macro_f1', 'bal_acc', 'qwk', 'mae_ord', 'recall_alta']:
    v3 = resultados_v3_ref['lstm_v3'].get(m, 0)
    v4 = resultados_v4['gru_v4_qwk'].get(m, 0)
    delta = v4 - v3
    print(f'  {m:<14}: lstm_v3={v3:.4f}  gru_v4={v4:.4f}  ({delta:+.4f})')

with mlflow.start_run(run_name='gru_v4_qwk'):
    mlflow.log_param('model_type',        'gru')
    mlflow.log_param('seq_len',           SEQ_LEN)
    mlflow.log_param('n_features',        len(FEAT_GRU))
    mlflow.log_param('units',             '64-32')
    mlflow.log_param('loss',              'qwk_loss')
    mlflow.log_param('alpha_type',        'sqrt')
    mlflow.log_param('batch_size',        512)
    mlflow.log_param('max_seq_train',     MAX_SEQ_TRAIN)
    mlflow.log_param('max_seq_val',       MAX_SEQ_VAL)
    mlflow.log_param('epochs_run',        len(history_gru.history['loss']))
    mlflow.log_param('dataset_version',   'v4')
    mlflow.log_param('d18_arch',          'lstm_to_gru')
    mlflow.log_param('d18_loss',          'focal_to_qwk')
    r = resultados_v4['gru_v4_qwk']
    mlflow.log_metric('val_macro_f1',    r['macro_f1'])
    mlflow.log_metric('val_bal_acc',     r['bal_acc'])
    mlflow.log_metric('val_qwk',         r['qwk'])
    mlflow.log_metric('val_mae_ordinal', r['mae_ord'])
    mlflow.log_metric('val_recall_alta', r['recall_alta'])
    mlflow.log_metric('val_prec_alta',   r['prec_alta'])
    mlflow.log_metric('val_candidato',   int(r['candidato_crit']))
print('GRU v4 (QWK Loss) registrado en MLflow.')


Memoria liberada para GRU.
Construyendo secuencias train (max=400,000)...
  X_train_seq: (400000, 4, 113)
Construyendo secuencias val (max=200,000)...
  X_val_seq:   (200000, 4, 113)


2026-07-28 07:29:51.544332: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 4, 113)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 4, 64)          │        34,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,964 (175.64 KB)

 Trainable params: 44,964 (175.64 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 18ms/step - accuracy: 0.8036 - loss: 0.5215 - val_accuracy: 0.7819 - val_loss: 0.8444 - learning_rate: 0.0010
Epoch 2/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.8131 - loss: 0.4660 - val_accuracy: 0.7923 - val_loss: 0.8109 - learning_rate: 0.0010
Epoch 3/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.7899 - loss: 0.4510 - val_accuracy: 0.8050 - val_loss: 0.8044 - learning_rate: 0.0010
Epoch 4/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.7894 - loss: 0.4429 - val_accuracy: 0.7927 - val_loss: 0.7941 - learning_rate: 0.0010
Epoch 5/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.8027 - loss: 0.4364 - val_accuracy: 0.7855 - val_loss: 0.7935 - learning_rate: 0.0010
Epoch 6/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.7961 - loss: 0.4345 - val_accuracy: 0.8101 - val_loss: 0.7860 - learning_rate: 0.0010
Epoch 7/50
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.7980 - l

In [5]:
import pickle

X_val_imp = val_imp[feat_cols_tab].fillna(0).values.astype('float32')

y_pred_base = val_imp[TARGET_COL].values.astype(int)
resultados_v4['baseline_v4'] = eval_model(y_val, y_pred_base, 'Baseline Estatico v4 — Validacion')

rf_v4 = pickle.load(open(f'{OUT_DIR}/rf_v4_full_model.pkl', 'rb'))
y_pred_rf_v4 = rf_v4.predict(X_val_imp)
resultados_v4['rf_v4_full'] = eval_model(y_val, y_pred_rf_v4, 'RF v4 Full — Validacion')
del rf_v4; gc.collect()

X_vl_nd = val_imp[feat_cols_no_d12].fillna(0).values.astype('float32')
rf_v4_nd = pickle.load(open(f'{OUT_DIR}/rf_v4_nod12_model.pkl', 'rb'))
y_pred_rf_nd = rf_v4_nd.predict(X_vl_nd)
resultados_v4['rf_v4_no_d12'] = eval_model(y_val, y_pred_rf_nd, 'RF v4 Sin-D12 — Validacion')
del rf_v4_nd, X_vl_nd; gc.collect()

xgb_v4 = xgb.XGBClassifier()
xgb_v4.load_model(f'{OUT_DIR}/xgb_v4_model.json')
X_val_xgb = val_imp[feat_cols_tab].values.astype('float32')
y_pred_xgb = xgb_v4.predict(X_val_xgb)
resultados_v4['xgb_v4'] = eval_model(y_val, y_pred_xgb, 'XGBoost v4 — Validacion')
del xgb_v4, X_val_xgb; gc.collect()

print("resultados_v4 keys:", list(resultados_v4.keys()))



  Baseline Estatico v4 — Validacion
  Macro-F1         : 0.6255  [Floor]
  Weighted-F1      : 0.8910
  Balanced Accuracy: 0.6254  [Floor]
  QWK (ordinal)    : 0.6149  [Bajo Floor]
  MAE ordinal      : 0.1655  [Stretch]
  Recall Alta      : 0.5598  [Floor]
  Precision Alta   : 0.5600  [Floor]
  -> Candidato (QWK>=0.65 AND RecAlta>=0.55): NO

  Matriz de confusion (filas=real, cols=pred):
        Normal   Baja  Media   Alta
Normal  521114  18853   5198   6026
Baja     18910  36552   2149   1034
Media     5282   2205   5509   1773
Alta      5946    978   1917  11244

  Reporte por clase:
              precision    recall  f1-score   support

      Normal       0.95      0.95      0.95    551191
        Baja       0.62      0.62      0.62     58645
       Media       0.37      0.37      0.37     14769
        Alta       0.56      0.56      0.56     20085

    accuracy                           0.89    644690
   macro avg       0.63      0.63      0.63    644690
weighted avg       0.89    

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.3s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.3s finished



  RF v4 Full — Validacion
  Macro-F1         : 0.6388  [Floor]
  Weighted-F1      : 0.8943
  Balanced Accuracy: 0.6471  [Floor]
  QWK (ordinal)    : 0.6257  [Bajo Floor]
  MAE ordinal      : 0.1647  [Stretch]
  Recall Alta      : 0.6466  [Floor]
  Precision Alta   : 0.5464  [Floor]
  -> Candidato (QWK>=0.65 AND RecAlta>=0.55): NO

  Matriz de confusion (filas=real, cols=pred):
        Normal   Baja  Media   Alta
Normal  521230  16900   5587   7474
Baja     18812  36777   1673   1383
Media     5191   2202   5452   1924
Alta      5120    564   1414  12987

  Reporte por clase:
              precision    recall  f1-score   support

      Normal       0.95      0.95      0.95    551191
        Baja       0.65      0.63      0.64     58645
       Media       0.39      0.37      0.38     14769
        Alta       0.55      0.65      0.59     20085

    accuracy                           0.89    644690
   macro avg       0.63      0.65      0.64    644690
weighted avg       0.89      0.89    

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.0s finished



  RF v4 Sin-D12 — Validacion
  Macro-F1         : 0.6436  [Floor]
  Weighted-F1      : 0.8950
  Balanced Accuracy: 0.6589  [Floor]
  QWK (ordinal)    : 0.6296  [Bajo Floor]
  MAE ordinal      : 0.1653  [Stretch]
  Recall Alta      : 0.6516  [Floor]
  Precision Alta   : 0.5393  [Floor]
  -> Candidato (QWK>=0.65 AND RecAlta>=0.55): NO

  Matriz de confusion (filas=real, cols=pred):
        Normal   Baja  Media   Alta
Normal  519738  17642   6011   7800
Baja     17918  37445   1849   1433
Media     4792   2083   5947   1947
Alta      4858    557   1582  13088

  Reporte por clase:
              precision    recall  f1-score   support

      Normal       0.95      0.94      0.95    551191
        Baja       0.65      0.64      0.64     58645
       Media       0.39      0.40      0.39     14769
        Alta       0.54      0.65      0.59     20085

    accuracy                           0.89    644690
   macro avg       0.63      0.66      0.64    644690
weighted avg       0.90      0.89 

In [6]:

W = 110
print('=' * W)
print('EVALUACION v4 vs UMBRALES — COMPARACION v3 → v4')
print('Criterio de candidatura: QWK >= 0.65 AND Recall Alta >= 0.55 (simultaneamente)')
print('Mejoras v4: D17 fillna | D18 GRU+QWK Loss | D19 colsample=0.6 | D20 dual D12-REV')
print('=' * W)
print(f'{"Modelo":<28} {"MacroF1":>8} {"BalAcc":>8} {"QWK":>8} '
      f'{"MAE":>8} {"RecAlta":>8} {"PrecAlta":>9} {"Candidato":>10}')
print(f'{"Floor (Benchmarking)":<28} {">= 0.55":>8} {">= 0.60":>8} {">= 0.65":>8} '
      f'{"<= 0.55":>8} {">= 0.55":>8} {">= 0.50":>9} {"QWK+Rec":>10}')
print('-' * W)

def print_row(nombre, r):
    if not r:
        print(nombre)
        return
    qwk  = r.get('qwk', 0)
    ra   = r.get('recall_alta', 0)
    cand = 'SI' if (qwk >= UMBRALES['qwk']['floor'] and ra >= UMBRALES['recall_alta']['floor']) else 'no'
    print(f'{nombre:<28} {r.get("macro_f1",0):>8.4f} {r.get("bal_acc",0):>8.4f} '
          f'{qwk:>8.4f} {r.get("mae_ord",0):>8.4f} {ra:>8.4f} '
          f'{r.get("prec_alta",0):>9.4f} {cand:>10}')

rows = [
    ('Baseline v3 (ref)',       resultados_v3_ref.get('baseline_v3', {})),
    ('RF v3 (ref)',             resultados_v3_ref.get('rf_v3',       {})),
    ('XGBoost v3 (ref)',        resultados_v3_ref.get('xgb_v3',      {})),
    ('LSTM v3 (ref)',           resultados_v3_ref.get('lstm_v3',      {})),
    ('── v4 (hoy) ─────────', {}),
    ('Baseline v4',             resultados_v4.get('baseline_v4',    {})),
    ('RF v4 Full',              resultados_v4.get('rf_v4_full',     {})),
    ('RF v4 Sin-D12 (ablac.)',  resultados_v4.get('rf_v4_no_d12',  {})),
    ('XGBoost v4',              resultados_v4.get('xgb_v4',         {})),
    ('GRU v4 (QWK Loss)',       resultados_v4.get('gru_v4_qwk',    {})),
]
for nombre, r in rows:
    print_row(nombre, r)

print('=' * W)
print(f'{"Target (Benchmarking)":<28} {">= 0.65":>8} {">= 0.70":>8} {">= 0.72":>8} '
      f'{"<= 0.45":>8} {">= 0.75":>8} {">= 0.62":>9}')
print(f'{"Stretch (Benchmarking)":<28} {">= 0.75":>8} {">= 0.78":>8} {">= 0.80":>8} '
      f'{"<= 0.35":>8} {">= 0.85":>8} {">= 0.75":>9}')

print('\n' + '=' * 70)
print('D12-REV — Decision sobre lag48_* y lag336_* para FE v5')
print('Protocolo: MDI individual >= 1/N_features  O  delta QWK full-noD12 > 0.005')
print('=' * 70)
try:
    thr  = 1.0 / len(feat_cols_tab)
    m48  = importances_rf_v4[[c for c in lag_d12_cols if c.startswith('lag48_')]].max()
    m336 = importances_rf_v4[[c for c in lag_d12_cols if c.startswith('lag336_')]].max()
    dqwk = (resultados_v4.get('rf_v4_full', {}).get('qwk', 0) -
            resultados_v4.get('rf_v4_no_d12', {}).get('qwk', 0))

    print(f'  Umbral MDI individual : {thr:.4f}  (1/{len(feat_cols_tab)})')
    print(f'  lag48_*  MDI max      : {m48:.4f}  -> {"H1" if m48 >= thr else "H0"}')
    print(f'  lag336_* MDI max      : {m336:.4f}  -> {"H1" if m336 >= thr else "H0"}')
    print(f'  Delta QWK full-noD12  : {dqwk:+.4f}  -> {"refuerza H1" if dqwk > 0.005 else ("refuerza H0" if dqwk < -0.005 else "marginal")}')
    print()
    for grp, val in [('lag48_*', m48), ('lag336_*', m336)]:
        decision = 'MANTENER en FE v5 (H1)' if val >= thr else 'ELIMINAR en FE v5 (H0)'
        print(f'  {grp:<12}: {decision}')
except NameError:
    print('[INFO] importances_rf_v4 no disponible — ejecutar Cell 4 primero')

print(f'\nModelos v4 guardados en: {OUT_DIR}')
candidatos = [k for k, v in resultados_v4.items() if v.get('candidato_crit', False)]
if candidatos:
    print(f'\nCANDIDATOS (criterio cumplido): {candidatos}')
    print('Proximos pasos: 06 evaluacion test set v2.ipynb')
else:
    print('\nNingun modelo alcanza criterio de candidatura en validacion.')
    print('Proximos pasos: revisar DECISION_LOG.md para ajuste de hiperparametros o FE v5.')


EVALUACION v4 vs UMBRALES — COMPARACION v3 → v4
Criterio de candidatura: QWK >= 0.65 AND Recall Alta >= 0.55 (simultaneamente)
Mejoras v4: D17 fillna | D18 GRU+QWK Loss | D19 colsample=0.6 | D20 dual D12-REV
Modelo                        MacroF1   BalAcc      QWK      MAE  RecAlta  PrecAlta  Candidato
Floor (Benchmarking)          >= 0.55  >= 0.60  >= 0.65  <= 0.55  >= 0.55   >= 0.50    QWK+Rec
--------------------------------------------------------------------------------------------------------------
Baseline v3 (ref)              0.6255   0.6254   0.6149   0.1655   0.5598    0.0000         no
RF v3 (ref)                    0.6436   0.6533   0.6324   0.1620   0.6361    0.0000         no
XGBoost v3 (ref)               0.6318   0.6968   0.6012   0.1956   0.6990    0.0000         no
LSTM v3 (ref)                  0.5984   0.6371   0.5386   0.2089   0.5741    0.0000         no
── v4 (hoy) ─────────
Baseline v4                    0.6255   0.6254   0.6149   0.1655   0.5598    0.5600      